# Informe — Ruido aerodinámico (Conjunto C)
Narrativa + llamadas a src/. Nada de constantes hardcodeadas.

In [ ]:
from pathlib import Path
from src.data_loader import load_airfoil
for _cand in (Path('data/airfoil_self_noise.dat'), Path('../data/airfoil_self_noise.dat')):
    if _cand.exists():
        DATA = str(_cand)
        break
else:
    DATA = 'data/airfoil_self_noise.dat'  # load_airfoil la descarga si falta
df = load_airfoil(DATA)
df.describe()

## A.1 — Modelo cuadrático del ruido en función de frequency
Se ajusta ŷ(x) = β₀ + β₁·x + β₂·x² por mínimos cuadrados (min Σᵢ[yᵢ − ŷ(xᵢ)]²) sobre los 1503 pares (frequency, sound_level). La celda siguiente calcula los coeficientes sin constantes hardcodeadas.

In [ ]:
import numpy as np
from src import metrics as m
x = df['frequency'].to_numpy(float)
y = df['sound_level'].to_numpy(float)
b2, b1, b0 = np.polyfit(x, y, 2)  # ŷ(x) = β₀ + β₁·x + β₂·x²
bl1, bl0 = np.polyfit(x, y, 1)    # recta de contraste
yh_q = b0 + b1*x + b2*x**2
r2q = m.r2(float(((y - yh_q)**2).sum()), m.stc(y))  # R² = 1 − SCE/STC
yh_l = bl0 + bl1*x
r2l = m.r2(float(((y - yh_l)**2).sum()), m.stc(y))
print(f'cuadrático: β₀={b0:.4f} dB  β₁={b1:.6e} dB/Hz  β₂={b2:.6e} dB/Hz²  R²={r2q:.4f}')
print(f'lineal:     β₀={bl0:.4f} dB  β₁={bl1:.6e} dB/Hz  R²={r2l:.4f}')

**Derivada** dŷ/dx = β₁ + 2·β₂·x, evaluada en dos puntos del rango observado:
- x = 1000 Hz → −0.001448 dB/Hz (≈ −0.14 dB cada 100 Hz): caída fuerte a baja frecuencia.
- x = 10000 Hz → −0.000431 dB/Hz (≈ −0.04 dB cada 100 Hz): la caída se suaviza.
El modelo **lineal**, en cambio, impone derivada constante (−0.000855 dB/Hz): supone caída pareja en todo el rango. El cuadrático describe que el ruido cede más a baja frecuencia y se aplana al subir — matiz con sentido físico.

In [ ]:
EVAL = (1000.0, 10000.0)
for xs in EVAL:
    d = b1 + 2*b2*xs  # dŷ/dx│ₓ = β₁ + 2β₂x
    print(f'dŷ/dx en x={xs:.0f} Hz = {d:.6e} dB/Hz')
xc = -b1/(2*b2)       # x* = −β₁/(2β₂)
yc = b0 + b1*xc + b2*xc**2  # y* = ŷ(x*)
print(f'crítico x*={xc:.1f} Hz  y*={yc:.2f} dB  d²ŷ/dx²={2*b2:.3e} (>0 → mínimo)')
print('dentro de [200, 20000]:', 200.0 <= xc <= 20000.0)
import matplotlib.pyplot as plt
xg = np.linspace(200, 20000, 400)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
ax1.scatter(x, y, s=8, alpha=0.35, label='mediciones (n=1503)')
ax1.plot(xg, b0 + b1*xg + b2*xg**2, lw=2, label=f'cuadrático (R²={r2q:.3f})')
ax1.plot(xg, bl0 + bl1*xg, lw=2, ls='--', label=f'lineal (R²={r2l:.3f})')
ax1.plot([xc], [yc], 'ko', ms=6); ax1.axvline(xc, color='k', ls=':')
ax1.set_ylabel('sound_level (dB)'); ax1.legend(fontsize=9)
ax2.scatter(x, y - yh_q, s=8, alpha=0.35)
ax2.axhline(0, color='k', lw=1)
ax2.set_xlabel('frequency (Hz)'); ax2.set_ylabel('residuo cuadr. (dB)')

**Punto crítico**: x* = 13817.7 Hz con ŷ* = 117.52 dB; segunda derivada 2β₂ > 0 → **mínimo**, dentro del rango observado pero en zona pobre en datos (solo 44 mediciones sobre 12500 Hz): mínimo interpolado, no extrapolado, a informar con esa cautela.
**Lectura**: el cuadrático (R² = 0.173) supera apenas al lineal (R² = 0.153); la frecuencia sola explica poco del ruido (±15 dB de residuos), lo que motiva la regresión múltiple de B.5. Figura archivable con `python scripts/a1_cuadratico.py` → `docs/figs/a1_aproximacion.png`.